# 04 — YouTube Music: đối chiếu metadata và log nhận diện

Phân tích **offline** log của mẫu đã được người dùng cho phép. Notebook không gọi mạng, không tải audio, không đọc/sửa DB ứng dụng. Chỉ các ID trong manifest được đưa vào bảng đối chiếu; không thêm bài từ danh sách gợi ý.

Mục đích: so nguồn `get_song.videoDetails.musicVideoType` với `get_watch_playlist.tracks[].videoType`, xác định tầng nhận trước và phần cần feature engineering. Đây là discovery, **không đo accuracy/recall toàn lịch sử**.

Nguồn: [get_song](https://ytmusicapi.readthedocs.io/en/stable/reference/browsing.html#ytmusicapi.YTMusic.get_song), [watch playlist](https://ytmusicapi.readthedocs.io/en/stable/reference/watch.html), [ý nghĩa videoType](https://ytmusicapi.readthedocs.io/en/stable/faq.html#which-videotypes-exist-and-what-do-they-mean). API không chính thức; kết quả là quan sát tại thời điểm/language/location trong manifest.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import collections, hashlib, json, os
import pandas as pd
from IPython.display import display

ROOT = Path.cwd().resolve()
if not (ROOT / 'pyproject.toml').exists():
    ROOT = ROOT.parent
RUN = Path(os.environ.get('AURALYTICA_YTM_RUN', str(ROOT / 'artifacts/ytmusic-pilot/20260911T041238Z'))).expanduser().resolve()
assert RUN.is_dir(), f'Không tìm thấy log local: {RUN}'
manifest = json.loads((RUN / 'manifest.json').read_text())
events = [json.loads(line) for line in (RUN / 'events.jsonl').read_text().splitlines() if line.strip()]
summary = json.loads((RUN / 'summary.json').read_text())
manifest_hash = hashlib.sha256((RUN / 'manifest.json').read_bytes()).hexdigest()
analysis_id = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
OUT = ROOT / 'artifacts/notebook-runs/04_ytmusic_audit' / RUN.name / analysis_id
OUT.mkdir(parents=True, exist_ok=False)
print('Run:', RUN.name, '| package:', manifest['package'], '| location:', manifest['location'])
print('Output:', OUT)

## 1. Kiểm tra tính đầy đủ và nguồn dữ liệu

Một observation cho mỗi phương thức/video trong pilot này. HTTP log có thêm request khởi tạo client. `observed` chỉ có nghĩa response chứa đúng ID, không phải kết luận là nhạc. Bản probe đã chạy được lưu cùng run để kiểm tra hash. Snapshot baseline lấy từ manifest đã duyệt, không phải trạng thái app hiện tại.

In [ ]:
sample = pd.DataFrame(manifest['samples']).rename(columns={'id':'video_id'})
obs = pd.DataFrame([e for e in events if e['event'] == 'observation'])
http = pd.DataFrame([e for e in events if e['event'] == 'http'])
ids = set(sample['video_id'])
assert sample['video_id'].is_unique
assert set(obs['video_id']) == ids
assert set(obs['provider_method']) == {'get_song','get_watch_playlist'}
assert not obs.duplicated(['video_id','provider_method']).any()
assert len(obs) == 2 * len(sample) == summary['method_calls']
assert len(sample) == summary['sample_videos']
assert all(e['run_id'] == manifest['run_id'] for e in events)
if (RUN / 'probe.py').exists():
    assert hashlib.sha256((RUN / 'probe.py').read_bytes()).hexdigest() == manifest['script_sha256']
if 'approved_manifest' in manifest:
    approved = RUN.parent / manifest['approved_manifest']
    assert hashlib.sha256(approved.read_bytes()).hexdigest() == manifest['approved_manifest_sha256']
    assert set(r['id'] for r in json.loads(approved.read_text())['samples']) == ids
print('Videos:', len(sample), '| observations:', len(obs), '| HTTP requests:', len(http))
display(sample.groupby('stratum').size().rename('videos').to_frame())
display(obs.groupby(['provider_method','status']).size().rename('observations').to_frame())

## 2. Bảng đối chiếu từng video

Giữ riêng loại từ player và queue; queue có thể gán UGC cho video không phải nhạc. Không dùng thời lượng ngắn làm bằng chứng chắc chắn là Shorts. `watch_count` là số lượt cá nhân trong Takeout; lượt xem công khai của YouTube không được nhập vào feature này.

In [ ]:
songs = obs[obs['provider_method'].eq('get_song')].set_index('video_id')
queues = obs[obs['provider_method'].eq('get_watch_playlist')].set_index('video_id')
audit = sample[['video_id','title','channel_name','watch_count','stratum','auto_group','auto_reason','user_group','metadata_json']].copy().set_index('video_id')
for name in ['music_video_type','category','playability','duration_seconds','status','exact_match','elapsed_ms']:
    audit['player_' + name] = songs[name] if name in songs else None

def exact_types(row):
    tracks = row.get('exact_tracks')
    if not isinstance(tracks, list):
        return []
    return sorted({t['videoType'] for t in tracks if t.get('videoId') == row.name and t.get('videoType')})

audit['queue_types'] = queues.apply(exact_types, axis=1)
audit['queue_status'] = queues['status']
audit['queue_exact_match'] = queues['exact_match']
audit['queue_elapsed_ms'] = queues['elapsed_ms']
audit['baseline_effective_group'] = audit['user_group'].fillna(audit['auto_group'])
audit['confirmed_music_by_user'] = audit['stratum'].eq('user_confirmed_music')
audit['url'] = 'https://www.youtube.com/watch?v=' + audit.index
cols = ['title','channel_name','watch_count','auto_group','auto_reason','player_music_video_type','category']
display(audit[['title','watch_count','auto_group','auto_reason','player_music_video_type','player_category','player_playability','queue_types']])

## 3. Độ phủ các giả thuyết — chưa áp dụng làm classifier

- Baseline: nhóm tự động `rules-v1` từ manifest.
- Queue có type: minh họa phương án quá rộng, không dùng nhận nhạc.
- Player strong: ATV/OMV/OFFICIAL_SOURCE_MUSIC là bằng chứng mạnh để thử; vẫn cần kiểm chứng Shorts/podcast/xung đột và giữ sửa tay.
- Player có type: chỉ là feature, **không tự coi mọi UGC là nhạc**.
- Category Music: kiểm tra vì sao không được dùng làm điều kiện bắt buộc.

Số trên 5 ca người dùng xác nhận là độ phủ trên nhóm đã biết bị bỏ sót, không phải recall tổng thể. Các nhóm Topic/library chỉ là proxy; nhóm đối chứng dựa trên tiêu đề/kênh chưa phải bộ nhãn độc lập.

In [ ]:
strong = {'MUSIC_VIDEO_TYPE_ATV','MUSIC_VIDEO_TYPE_OMV','MUSIC_VIDEO_TYPE_OFFICIAL_SOURCE_MUSIC'}
exact = audit['player_status'].eq('observed') & audit['player_exact_match'].eq(True)
gates = pd.DataFrame(index=audit.index)
gates['baseline_music'] = audit['auto_group'].eq('music')
gates['queue_has_type'] = audit['queue_types'].map(bool) & audit['queue_exact_match'].eq(True)
gates['player_strong_type'] = exact & audit['player_music_video_type'].isin(strong)
gates['player_has_type'] = exact & audit['player_music_video_type'].notna()
gates['player_category_music'] = exact & audit['player_category'].eq('Music')
coverage = pd.DataFrame({
    'matched_in_sample': gates.sum(),
    'matched_user_confirmed_music': gates.loc[audit['confirmed_music_by_user']].sum(),
    'matched_other_strata': gates.loc[~audit['confirmed_music_by_user']].sum(),
})
display(coverage)
residual = audit[audit['confirmed_music_by_user'] & ~gates['player_strong_type']]
print('Nhạc đã xác nhận còn cần xử lý sau strong gate:', len(residual))
display(residual[['title','watch_count','player_music_video_type','player_category','auto_reason']])
# Separate the residual's priority from the decision to auto-select it.
audit['research_stage'] = 'residual_unknown'
audit.loc[gates['player_strong_type'],'research_stage'] = 'strong_music_evidence'
audit.loc[exact & audit['player_music_video_type'].eq('MUSIC_VIDEO_TYPE_UGC'),'research_stage'] = 'ugc_needs_combined_evidence'
audit.loc[~exact,'research_stage'] = 'metadata_missing_or_error'
audit['manual_label'] = ''
audit.loc[audit['confirmed_music_by_user'],'manual_label'] = 'music'
audit['label_source'] = ''
audit.loc[audit['confirmed_music_by_user'],'label_source'] = 'user_confirmed_before_pilot'
audit['notes'] = '' 

## 4. Thời gian và giới hạn thử nghiệm

Thời gian phương thức gồm chi phí client/request; wall time trong summary bao gồm khoảng nghỉ giữa các lần gọi nhưng không gồm khởi tạo client. Cache tắt, một attempt/phương thức. Không suy diễn thành SLA hoặc hứa thời gian cho toàn lịch sử.

Mẫu thiếu podcast thật, ca BGM được nền tảng nhận là music, video xóa/chặn vùng và Shorts đã xác minh. Nhóm library cũng đều là Topic; ba ca kênh giải trí cùng một kênh. Chưa đo false positives của player UGC, chưa chốt ngưỡng recurrence. Mẫu công khai trước đó có ATV nhưng UNPLAYABLE, nên không dùng playability làm nhãn nội dung hay kết luận chắc chắn tải được.

In [ ]:
latency = obs.groupby('provider_method')['elapsed_ms'].agg(['count','min','median','max','sum'])
display(latency)
print('Wall time (không gồm khởi tạo):', summary['elapsed_seconds'], 'seconds')
print('HTTP statuses:', http['http_status'].value_counts(dropna=False).to_dict())
print('Method statuses:', obs['status'].value_counts(dropna=False).to_dict())

## 5. Xuất đối chiếu và vòng phản hồi

Bảng review chỉ điền sẵn 5 nhãn người dùng đã xác nhận trước pilot; không biến output YTM thành ground truth. Điền các nhãn còn lại bằng music/non_music/uncertain/unavailable và lý do sau khi duyệt. Mỗi lần chạy notebook ghi thư mục mới, không ghi đè review đã điền.

Log ứng dụng cần bổ sung sau pilot: run/snapshot/version/config, observation có source path và cache age, feature snapshot, quyết định trước/sau và override, user correction liên kết decision ID. Lưu thiếu metadata/lỗi truy vấn riêng với non-music; ghi log append-only, không chỉ cập nhật evidence_json hiện tại.

In [ ]:
audit.to_csv(OUT / 'video_audit.csv')
gates.to_csv(OUT / 'hypothesis_matches.csv')
coverage.to_csv(OUT / 'hypothesis_coverage.csv')
latency.to_csv(OUT / 'latency.csv')
audit[['title','channel_name','url','watch_count','research_stage','manual_label','label_source','notes']].to_csv(OUT / 'review.csv')
result = dict(run_id=manifest['run_id'], manifest_sha256=manifest_hash,
              observation_log_sha256=hashlib.sha256((RUN / 'events.jsonl').read_bytes()).hexdigest(),
              analysis_version='ytm-audit-v1', sample_videos=len(audit), method_calls=len(obs),
              confirmed_music=int(audit['confirmed_music_by_user'].sum()),
              confirmed_music_with_player_type=int(gates.loc[audit['confirmed_music_by_user'],'player_has_type'].sum()),
              confirmed_music_strong_type=int(gates.loc[audit['confirmed_music_by_user'],'player_strong_type'].sum()),
              confirmed_music_residual=len(residual),
              counts=gates.sum().astype(int).to_dict(),
              warning='Discovery only; no population precision/recall or production changes.')
(OUT / 'analysis_summary.json').write_text(json.dumps(result, ensure_ascii=False, indent=2))
print(json.dumps(result, ensure_ascii=False, indent=2))